# Style2Fit — Step 1: Generate Training Data
Converts Fashion-Gen dataset into casual situation → structured outfit pairs for LLM fine-tuning.

**Runtime:** CPU is fine. Takes ~20 min for 500 pairs.

In [ ]:
!pip install openai datasets tqdm -q

In [ ]:
import os
from google.colab import userdata

# Add OPENAI_API_KEY via Colab Secrets (lock icon in left sidebar)
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
import json
import random
import re
from pathlib import Path
from openai import OpenAI
from datasets import load_dataset

random.seed(42)
Path('data').mkdir(exist_ok=True)

client = OpenAI()

REQUIRED_FIELDS = ['Top:', 'Bottom:', 'Shoes:', 'Aesthetic:', 'Explanation:']

CATEGORY_SIGNALS = {
    'top':       ['shirt', 'tee', 'blouse', 'sweater', 'top', 'jacket', 'coat',
                  'blazer', 'cardigan', 'hoodie', 'sweatshirt', 'vest', 'tank'],
    'bottom':    ['pant', 'trouser', 'jean', 'skirt', 'short', 'legging', 'chino',
                  'slack', 'culotte', 'wide-leg', 'straight-leg'],
    'dress':     ['dress', 'gown', 'romper', 'jumpsuit', 'bodysuit', 'mini', 'midi', 'maxi'],
    'shoes':     ['shoe', 'boot', 'sneaker', 'heel', 'loafer', 'sandal', 'flat',
                  'oxford', 'pump', 'mule', 'wedge'],
    'outerwear': ['coat', 'jacket', 'blazer', 'trench', 'parka', 'anorak',
                  'overcoat', 'windbreaker', 'cape'],
    'accessory': ['bag', 'belt', 'scarf', 'hat', 'earring', 'necklace', 'bracelet',
                  'sunglasses', 'watch', 'purse', 'tote', 'clutch'],
}

EXTRA_SEEDS = [
    # --- dress code confusion ---
    "wedding invite says cocktail attire what does that even mean",
    "work event says smart casual i have no idea what to wear",
    "black tie optional — do i actually need a gown",
    "the dress code is business casual but its a startup so idk",
    "garden party dress code, what do i wear",
    "invite says 'festive attire' what is that",
    "semi formal?? is that just like a nice dress or more",
    "resort casual for a bachelorette trip what does that mean",

    # --- body / fit context ---
    "i'm petite and everything is always too long, what should i wear to a dinner",
    "i have a longer torso, going to a rooftop bar, what works",
    "plus size, want something cute for a first date that isn't shapeless",
    "i'm tall and pants are never long enough, job interview tmrw help",
    "curvy, want to look put together for a work presentation",
    "apple shaped, want to look good at a wedding, not sure what flatters me",
    "i carry weight in my midsection, going to a summer party, help",

    # --- weather specificity ---
    "it's 45 degrees and windy, have a dinner reservation",
    "humid and like 85 degrees, outdoor wedding this weekend",
    "it's raining but i still need to look nice for a date",
    "cold enough for a coat but i'll be inside most of the time",
    "it's snowing but i have a holiday party tonight",
    "it's hot but the restaurant is always freezing inside",
    "transitional weather like 60s during the day but cold at night",

    # --- budget awareness ---
    "i'm a broke college student but have a formal event",
    "need to look professional on a budget, interview tomorrow",
    "want to look expensive without spending a lot",

    # --- versatility / rewearing ---
    "need one outfit i can wear to work and then go out after",
    "packing light for a weekend trip, need outfits that mix and match",
    "want something i can dress up or down depending on the occasion",

    # --- gender expression ---
    "i'm nonbinary and want something androgynous for a dinner date",
    "want a more masculine look for a night out",
    "going for an androgynous office look, any ideas",
    "i present masc but want something fashion forward for a party",
    "nonbinary, going to a wedding, don't want to wear a dress or a suit, help",
    "i want something gender neutral for a first date",
]

# Multi-outfit trip seeds — output is a 3-day capsule wardrobe, not a single outfit
TRIP_SEEDS = [
    ("5 days in barcelona in april, mix of sightseeing and nice dinners", 5),
    ("weekend trip to nyc in november, need looks that mix and match", 3),
    ("week in paris in fall, want to look chic but be comfortable walking", 5),
    ("3 days in miami, mix of beach and going out at night", 3),
    ("long weekend in nashville, mix of country bars and nice restaurants", 3),
    ("4 days in tokyo, want to look stylish and appropriate", 4),
]

SYSTEM_PROMPT = """You are generating training data for a fashion AI assistant called Style2Fit.

Given a fashion product description, output TWO things:

1. SITUATION: A casual, conversational prompt a real person might type — like texting a friend.
   - Vary the register: sometimes lowercase, sometimes with typos, sometimes incomplete
   - Occasionally incorporate: dress code confusion, body/fit mentions, weather specifics, budget constraints, or gender expression
   - Examples: "coffee date tmrw help", "wedding says cocktail attire what does that mean", "i'm petite and have a dinner tonight"

2. OUTFIT: The structured outfit plan derived from the product description.
   - Fill in any missing pieces logically
   - If the outfit is a dress/jumpsuit/romper, Top should be "n/a" and Bottom should be the dress description
   - If situation mentions fit/body type, acknowledge it in the Explanation
   - If situation mentions dress code, clarify it in the Explanation
   - Keep descriptions specific: color, material, fit where available
   - Aesthetic should be 1-2 words
   - Explanation should be 2 sentences max, conversational

Always respond in EXACTLY this format, no extra text:
SITUATION: <casual prompt>
---
Top: <item or "n/a">
Bottom: <item or dress description>
Shoes: <item>
Outerwear: <item or "none needed">
Accessories: <2-3 items>
Aesthetic: <1-2 words>
Explanation: <why this works>"""

print('Setup complete.')

In [ ]:
def is_full_outfit(entry):
    description = entry.get('text') or entry.get('description') or entry.get('caption') or ''
    if len(description) < 40:
        return False
    desc_lower = description.lower()

    # A dress/jumpsuit/romper counts as a full outfit by itself — only needs shoes/accessories
    has_dress = any(s in desc_lower for s in CATEGORY_SIGNALS['dress'])
    if has_dress:
        return True

    # Otherwise require at least 3 categories (top + bottom + shoes/accessory)
    categories_found = sum(
        1 for signals in CATEGORY_SIGNALS.values()
        if any(s in desc_lower for s in signals)
    )
    return categories_found >= 3


def convert_entry(description):
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            max_tokens=400,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': f'Fashion description:\n{description}'},
            ],
        )
        text = response.choices[0].message.content.strip()

        if '---' not in text or 'SITUATION:' not in text:
            return None

        parts = text.split('---', 1)
        situation = re.sub(r'^SITUATION:\s*', '', parts[0].strip(), flags=re.IGNORECASE).strip()
        outfit_text = parts[1].strip()

        if not all(f in outfit_text for f in REQUIRED_FIELDS):
            return None

        return {
            'instruction': situation,
            'input': '',
            'output': outfit_text,
            'source': 'fashion200k',
        }
    except Exception as e:
        print(f'  Error: {e}')
        return None


print('Functions defined.')

In [6]:
N = 500  # number of training pairs to generate

print('Loading fashion200k dataset...')
ds = load_dataset('Marqo/fashion200k', split='data')
print(f'  Total entries: {len(ds)}')
print(f'  Fields: {ds.column_names}')

# Use dataset's built-in filter — much faster than iterating in Python
print('Filtering to full outfits...')
full_outfits = ds.filter(is_full_outfit, num_proc=2)
print(f'  Full outfit entries: {len(full_outfits)} ({len(full_outfits)/len(ds)*100:.1f}%)')

sampled_indices = random.sample(range(len(full_outfits)), min(N * 2, len(full_outfits)))
sampled = full_outfits.select(sampled_indices)
print(f'  Sampled {len(sampled)} to convert (2x target to account for failures)')

Loading fashion200k dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


  Total entries: 201624
  Fields: ['image', 'category1', 'category2', 'category3', 'text', 'item_ID']
Filtering to full outfits...
  Full outfit entries: 16184 (8.0%)
  Sampled 1000 to convert (2x target to account for failures)


In [ ]:
from tqdm.notebook import tqdm
import os

os.makedirs('data', exist_ok=True)
out_path = 'data/train.jsonl'

# Resume from existing file if session reconnected
pairs = []
if os.path.exists(out_path):
    with open(out_path) as f:
        pairs = [json.loads(l) for l in f if l.strip()]
    print(f'Resuming — {len(pairs)} pairs already saved')

N = 500
entries_to_process = list(sampled)[:N * 2]
attempts = 0

progress = tqdm(total=N, initial=len(pairs), desc='Generating pairs', unit='pair')

# Open in append mode so each pair is saved immediately
with open(out_path, 'a') as f:
    for entry in entries_to_process:
        if len(pairs) >= N:
            break
        description = entry.get('text') or entry.get('description') or entry.get('caption') or ''
        if not description:
            continue
        attempts += 1
        result = convert_entry(description)
        if result:
            pairs.append(result)
            f.write(json.dumps(result) + '\n')
            f.flush()  # write to disk immediately
            progress.update(1)
            progress.set_postfix({
                'attempts': attempts,
                'success_rate': f'{len(pairs)/attempts*100:.0f}%',
            })

progress.close()
print(f'\nDone. {len(pairs)} pairs saved to {out_path}')

In [ ]:
SEED_SYSTEM_PROMPT = """You are generating training data for a fashion AI assistant called Style2Fit.

Given a situation prompt, generate a complete structured outfit recommendation.

Always respond in EXACTLY this format:
Top: <item or "n/a" if dress/jumpsuit>
Bottom: <item or dress description>
Shoes: <item>
Outerwear: <item or "none needed">
Accessories: <2-3 items>
Aesthetic: <1-2 words>
Explanation: <2 sentences — address fit/dress code/weather/gender if mentioned>"""

TRIP_SYSTEM_PROMPT = """You are generating training data for a fashion AI assistant called Style2Fit.

Given a multi-day trip situation, generate a capsule wardrobe: one outfit per day that shares key pieces so everything mixes and matches.

Always respond in EXACTLY this format (repeat the Day block for each day):
Day 1:
Top: ...
Bottom: ...
Shoes: ...
Outerwear: ...
Accessories: ...

Day 2:
...

Shared pieces: <list 2-3 items that appear across multiple days>
Aesthetic: <1-2 words>
Packing tip: <one sentence on how to pack light with these pieces>"""


def generate_from_seed(situation):
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            max_tokens=400,
            messages=[
                {'role': 'system', 'content': SEED_SYSTEM_PROMPT},
                {'role': 'user', 'content': situation},
            ],
        )
        outfit_text = response.choices[0].message.content.strip()
        if not all(f in outfit_text for f in REQUIRED_FIELDS):
            return None
        return {'instruction': situation, 'input': '', 'output': outfit_text, 'source': 'seed'}
    except Exception as e:
        print(f'  Error: {e}')
        return None


def generate_trip(situation, n_days):
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            max_tokens=800,
            messages=[
                {'role': 'system', 'content': TRIP_SYSTEM_PROMPT},
                {'role': 'user', 'content': f'{situation} ({n_days} days)'},
            ],
        )
        outfit_text = response.choices[0].message.content.strip()
        if 'Day 1:' not in outfit_text:
            return None
        return {'instruction': situation, 'input': f'{n_days} days', 'output': outfit_text, 'source': 'trip'}
    except Exception as e:
        print(f'  Error: {e}')
        return None


# --- Run seed generation ---
seed_pairs = []
print(f'Generating {len(EXTRA_SEEDS)} seed pairs (dress codes, body type, weather, gender)...')
for situation in tqdm(EXTRA_SEEDS, desc='Seed pairs', unit='pair'):
    result = generate_from_seed(situation)
    if result:
        seed_pairs.append(result)

# --- Run trip generation ---
trip_pairs = []
print(f'\nGenerating {len(TRIP_SEEDS)} trip capsule wardrobes...')
for situation, n_days in tqdm(TRIP_SEEDS, desc='Trip pairs', unit='trip'):
    result = generate_trip(situation, n_days)
    if result:
        trip_pairs.append(result)

# --- Combine all ---
all_pairs = pairs + seed_pairs + trip_pairs
print(f'\nTotal: {len(all_pairs)} pairs')
print(f'  fashion200k: {len(pairs)}')
print(f'  seed (dress codes / body / weather / gender): {len(seed_pairs)}')
print(f'  trip capsule wardrobes: {len(trip_pairs)}')

In [ ]:
import os, json
from google.colab import files

os.makedirs('data', exist_ok=True)

# Gather whatever exists in memory
all_saved = []
for var_name in ['pairs', 'seed_pairs', 'trip_pairs']:
    val = globals().get(var_name, [])
    if val:
        print(f'{var_name}: {len(val)} pairs in memory')
        all_saved.extend(val)
    else:
        print(f'{var_name}: empty or not defined')

if not all_saved:
    print('\nNothing in memory — session was lost. Re-run from cell-5.')
else:
    out_path = 'data/train.jsonl'
    with open(out_path, 'w') as f:
        for pair in all_saved:
            f.write(json.dumps(pair) + '\n')
    print(f'\nSaved {len(all_saved)} pairs to {out_path}')
    files.download(out_path)